# Lecture 12 examples: Approximate Bayesian computation

STAT-S681 · Simulation-Based Inference

# First example: normal location model

$\theta \sim \text{Uniform}[-3,3]$, $X \mid \theta \sim N(\theta, 1)$. Suppose we observe $x_{\text{obs}} = 0.8$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(681)

def simulate_normal():
    theta = rng.uniform(-3, 3)
    X = rng.normal(theta, 1)
    return {"theta": theta, "X": X}

Try rejection sampling.

In [ ]:
def rejection_sample_normal(
        x_obs=0.8):
    while True:
        run = simulate_normal()
        if run["X"] == x_obs:
            return run["theta"]

X = np.array(
    [simulate_normal()["X"]
     for _ in range(100_000)])
print(np.sum(X == 0.8))   # 0

Doesn't work! $P(X = 0.8) = 0$, so `rejection_sample_normal()` never returns. (Don't run it.)

# Approximate Bayesian computation

Keep $\theta^*$ if $d(S(X^*), S(x_{\text{obs}})) < \epsilon$. Take $S(x) = x$ and $d(x,y) = |x - y|$.

In [ ]:
def abc_sample_normal(eps, x_obs=0.8):
    while True:
        run = simulate_normal()
        if abs(run["X"] - x_obs) < eps:
            return run["theta"]

# What does $\epsilon$ do?

Compare $\epsilon = 1.5$, $0.2$, $0.05$. The orange line is the exact posterior.

In [ ]:
from scipy.stats import norm

eps_values = [1.5, 0.2, 0.05]
draws = [np.array([abc_sample_normal(eps) for _ in range(2000)])
         for eps in eps_values]

grid = np.linspace(-3, 3, 1001)
exact = norm.pdf(grid, 0.8, 1) / (norm.cdf(3, 0.8, 1) - norm.cdf(-3, 0.8, 1))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, eps, dr in zip(axes, eps_values, draws):
    ax.hist(dr, bins=np.arange(-3, 3.01, 0.2), density=True)
    ax.plot(grid, exact, color="orange", lw=2)
    ax.set_ylim(0, 0.5); ax.set_xlabel("theta"); ax.set_title(f"Tolerance {eps}")
plt.show()

Compare the tolerances with Q–Q plots.

In [ ]:
probs = np.linspace(0.005, 0.995, 199)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (i, j) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
    ax.scatter(np.quantile(draws[i], probs), np.quantile(draws[j], probs), s=8)
    ax.plot([-3, 3], [-3, 3], "k--")
    ax.set_xlabel(f"epsilon = {eps_values[i]}"); ax.set_ylabel(f"epsilon = {eps_values[j]}")
plt.show()

# What if we have twenty measurements?

A direct extension retains a dataset only when every simulated measurement is close to its observed counterpart: $\max_i |X_i-x_{\mathrm{obs},i}|<\epsilon$.

In [ ]:
rng = np.random.default_rng(681)
x_obs = rng.normal(0.8, 1, 20)   # observed data
X = rng.normal(0.8, 1, 20)       # simulated at theta = 0.8

def plot_match(x_obs, X, eps):
    near = np.abs(X - x_obs) < eps
    i = np.arange(1, len(x_obs) + 1)
    plt.errorbar(i, x_obs, yerr=eps, fmt="x", color="black")
    plt.scatter(i, X, color=np.where(near, "darkgreen", "gray"))
    plt.xlabel("measurement i"); plt.ylabel("value")
    plt.title(f"{near.sum()} of {len(X)} within {eps}")
    plt.show()
plot_match(x_obs, X, eps=0.5)

Each simulated measurement has to land inside its own bar.

# How might we check that the sample mean is sufficient?

We will compare two summaries of the twenty measurements: $S(X)$ = mean of all twenty (sufficient) and $S(X)$ = mean of the first ten (not sufficient).

Row $i$ of `X` is twenty measurements simulated with $\theta_i$. Table rows: $\theta$ above its median; columns: $a^\top X$ above its median.

In [ ]:
rng = np.random.default_rng(682)
n, N = 20, 200_000
theta = rng.uniform(-3, 3, N)
X = rng.normal(theta[:, None], 1, (N, n))
def check_summary(S_X, S_x_obs, width=0.05):
    keep = np.abs(S_X - S_x_obs) < width  # S(X) near S(x_obs)
    a = rng.normal(size=n)              # random direction
    proj = X[keep] @ a                  # projection a'X
    th = theta[keep]
    t_hi, p_hi = th > np.median(th), proj > np.median(proj)
    return np.histogram2d(t_hi, p_hi, bins=2)[0] / keep.sum()

print(check_summary(X.mean(axis=1), x_obs.mean()))
print(check_summary(X[:, :10].mean(axis=1), x_obs[:10].mean()))

Four random directions: plot $\theta$ against $a^\top X$ for the kept runs, using the same four directions for both summaries. Orange lines are the medians; the numbers are the fractions of kept runs in each box.

In [ ]:
A = rng.normal(size=(n, 4))   # four random directions
def plot_directions(S_X, S_x_obs, A, axes, width=0.05):
    keep = np.abs(S_X - S_x_obs) < width
    th = theta[keep]
    for k, ax in enumerate(axes):
        proj = X[keep] @ A[:, k]
        ax.scatter(proj, th, s=1, color="gray")
        ax.axvline(np.median(proj), color="orange", lw=2)
        ax.axhline(np.median(th), color="orange", lw=2)
        hi_t, hi_p = th > np.median(th), proj > np.median(proj)
        for x, y, frac in [(0.03, 0.95, np.mean(~hi_p & hi_t)), (0.97, 0.95, np.mean(hi_p & hi_t)),
                           (0.03, 0.03, np.mean(~hi_p & ~hi_t)), (0.97, 0.03, np.mean(hi_p & ~hi_t))]:
            ax.text(x, y, f"{frac:.2f}", transform=ax.transAxes,
                    ha="left" if x < 0.5 else "right", va="top" if y > 0.5 else "bottom")
        ax.set_xlabel(f"a'X, direction {k + 1}"); ax.set_ylabel("theta")

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
plot_directions(X.mean(axis=1), x_obs.mean(), A, axes[0])                # top: mean of all 20
plot_directions(X[:, :10].mean(axis=1), x_obs[:10].mean(), A, axes[1])   # bottom: mean of first 10
plt.tight_layout(); plt.show()

Top: every box holds 1/4 of the runs, in every direction. Bottom: the boxes are uneven by different amounts in different directions.

Many random directions: correlations between $\theta$ and $a^\top X$ across 200 random directions. Dashed lines mark $\pm 2/\sqrt{\text{number of runs}}$.

In [ ]:
def direction_cors(S_X, S_x_obs, width=0.05, K=200):
    keep = np.abs(S_X - S_x_obs) < width
    proj = X[keep] @ rng.normal(size=(n, K))   # K random directions
    th = theta[keep]
    r = [np.corrcoef(th, proj[:, k])[0, 1] for k in range(K)]
    return np.array(r), keep.sum()

summaries = {"Mean of all 20": (X.mean(axis=1), x_obs.mean()),
             "Mean of first 10": (X[:, :10].mean(axis=1), x_obs[:10].mean())}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (name, (S_X, S_x_obs)) in zip(axes, summaries.items()):
    r, n_kept = direction_cors(S_X, S_x_obs)
    ax.hist(r, bins=np.arange(-0.6, 0.601, 0.025))
    for b in (-2, 2):
        ax.axvline(b / np.sqrt(n_kept), ls="--", color="k")
    ax.set_title(name); ax.set_xlabel("correlation of theta with a'X")
plt.show()

# A learned summary for the rain model

Candidate features: rainy-day count, weather changes, endpoint weather, and longest rainy run. Use simulations to learn how these features jointly predict $\theta$.

In [ ]:
def rain_features(N, days=6):
    theta = rng.uniform(size=N)
    X = np.zeros((N, days), dtype=int)
    X[:, 0] = rng.binomial(1, theta)
    for d in range(1, days):
        redraw = rng.uniform(size=N) >= 0.5   # otherwise weather persists
        X[:, d] = np.where(redraw, rng.binomial(1, theta), X[:, d - 1])
    longest = np.zeros(N, dtype=int); current = np.zeros(N, dtype=int)
    for d in range(days):
        current = np.where(X[:, d] == 1, current + 1, 0)
        longest = np.maximum(longest, current)
    features = np.column_stack([np.ones(N), X.sum(axis=1),
                                (X[:, 1:] != X[:, :-1]).sum(axis=1),
                                X[:, 0], X[:, -1], longest])
    return theta, features

names = ["intercept", "count", "changes", "first", "last", "longest"]
rng = np.random.default_rng(687)
theta_train, Z_train = rain_features(150_000)
coef = np.linalg.lstsq(Z_train, theta_train, rcond=None)[0]
{k: round(float(v), 3) for k, v in zip(names, coef)}

The learned summary $S(X)$ is the fitted prediction. For DRRRRD:

In [ ]:
z_obs = np.array([1, 4, 2, 0, 0, 4])   # intercept, count, changes, first, last, longest
print(z_obs @ coef)

The learned summary on held-out simulations.

In [ ]:
theta_test, Z_test = rain_features(50_000)
pred = Z_test @ coef
show = rng.choice(len(pred), 5000, replace=False)
plt.figure(figsize=(5, 5))
plt.scatter(pred[show], theta_test[show], s=2)
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("Learned summary / prediction"); plt.ylabel("Generating theta")
plt.show()

# Regression adjustment

Twenty measurements summarized by their mean, with $S(x_{\text{obs}}) = 0.8$ and a wide tolerance of 0.5. The mean of twenty $N(\theta, 1)$ measurements is $N(\theta, 1/20)$, so we simulate it directly.

In [ ]:
rng = np.random.default_rng(690)
N = 1_000_000
theta = rng.uniform(-3, 3, N)
S_X = rng.normal(theta, 1 / np.sqrt(20))

eps = 0.5
keep = np.abs(S_X - 0.8) < eps
theta_kept = theta[keep]
d = S_X[keep] - 0.8                     # d_i = S(X_i) - S(x_obs)
w = 1 - (d / eps) ** 2                  # closer summaries get more weight

Fit $\theta_i=\alpha+\beta\,d_i+e_i$ to the kept draws and use $\theta_i^{\mathrm{adj}}=\widehat\alpha+\widehat e_i = \theta_i - \widehat\beta\, d_i$.

In [ ]:
beta_hat, alpha_hat = np.polyfit(d, theta_kept, 1, w=np.sqrt(w))   # weighted least squares
theta_adj = theta_kept - beta_hat * d

Compare with tolerance 0.05, unadjusted. The orange curve is the exact posterior.

In [ ]:
theta_narrow = theta[np.abs(S_X - 0.8) < 0.05]
exact = norm.pdf(grid, 0.8, 1 / np.sqrt(20))

panels = {"Tolerance 0.5, unadjusted": (theta_kept, None),
          "Tolerance 0.5, adjusted": (theta_adj, w),
          "Tolerance 0.05, unadjusted": (theta_narrow, None)}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, (v, wt)) in zip(axes, panels.items()):
    ax.hist(v, bins=np.arange(-1, 2.6, 0.05), weights=wt, density=True)
    ax.plot(grid, exact, color="orange", lw=2)
    ax.set_xlim(-0.6, 2.2); ax.set_ylim(0, 2); ax.set_xlabel("theta"); ax.set_title(name)
plt.show()

def wsd(v, wt):
    m = np.average(v, weights=wt)
    return np.sqrt(np.average((v - m) ** 2, weights=wt))
print("unadjusted", theta_kept.std(), "adjusted", wsd(theta_adj, w), "narrow", theta_narrow.std())